# Rubric Testing and Evaluation

This notebook demonstrates how to:
1. Test individual rubrics with sample data
2. Use the LangSmith wrapper for tracking rubric performance


In [1]:
%load_ext autoreload
%autoreload 2

In [74]:
import pandas as pd

from explain.eval.rubrics import CorrectnessRubric, FalsificationRubric, PlausibilityRubric
from explain.eval.utils import as_verifier_dataset

df = pd.read_csv("../../data/curation_v1/results/structure-explain-results-v3-claude4.csv")
df["response"] = df["raw_response"].apply(lambda x: x.split("</think>")[1].strip())
# sort df by the length of the "dag" column
df = df.sort_values(by="dag", key=lambda x: x.str.len(), ascending=True)
dt = as_verifier_dataset(df, answer_columns=["response"])

Map: 100%|██████████| 176/176 [00:00<00:00, 9193.57 examples/s]


# Rubric Testing and Evaluation

Let's test the individual rubrics to understand their behavior.

In [75]:
example_dt = dt.select(range(10))
example_dt.column_names

['question', 'answer', 'info', 'task']

In [76]:
example_dt["answer"][0]

'<answer>\nOclacitinib competitively binds the ATP-binding pocket of JAK1 (and JAK2), directly inhibiting their catalytic activity. In the context of SOCS3 knockdown, which removes a key negative feedback regulator of JAK-STAT signaling, this creates competing effects on the pathway. SOCS3 deficiency normally leads to enhanced and prolonged STAT3 phosphorylation and nuclear translocation, resulting in upregulated inflammatory gene expression and cytokine production. However, Oclacitinib\'s direct enzymatic inhibition of JAK1 counteracts this hyperactivation by blocking upstream kinase activity. The net result is partial restoration of normal JAK-STAT signaling dynamics and reduced inflammatory responses compared to SOCS3 knockdown alone, creating a distinctive transcriptomic signature with suppressed STAT3-dependent gene expression and normalized cellular behaviors.\n</answer>\n\n<explain>\nset_context(cell_type="immune cell", disease_context="pathway biology modulation", prior_perturb

In [77]:
# correctness rubric
correctness_rubric = CorrectnessRubric()
correctness_rubric.judge(
    prompt=example_dt["question"][0],
    completion=example_dt["answer"][0],
    answer=example_dt["answer"][0],
    state={},
)

2025-08-06 15:57:09.974 | INFO     | explain.llm._client:__init__:694 - Initialized unified LLM client with provider: anthropic


{'correctness': 1.0}

In [78]:
# plausibility rubric
plausibility_rubric = PlausibilityRubric()
plausibility_rubric.judge(
    prompt=example_dt["question"][0],
    completion=example_dt["answer"][0],
    answer=example_dt["answer"][0],
    state={},
)

2025-08-06 15:57:11.860 | INFO     | explain.llm._client:__init__:694 - Initialized unified LLM client with provider: anthropic


{'scientific_accuracy': 0.9,
 'logical_consistency': 0.9,
 'mechanistic_clarity': 0.8}

In [87]:
example_dt["question"][1]

"How does the following perturbation influence the cell in the described context, mechanistically and functionally?**  \n\n{'context': {'perturbation_type': 'soluble factor', 'description': 'Soluble factor addition of IL-13', 'cell_type': 'N/A', 'disease_model': 'Asthma and allergy - IL-13'}, 'perturbation': {'type': 'chemical', 'smiles': 'CC1=CN=C(N=C1NC2=CC(=CC=C2)S(=O)(=O)NC(C)(C)C)NC3=CC=C(C=C3)OCCN4CCCC4', 'name': 'TG 101348', 'target': 'JAK', 'moa_type': 'inhibitor'}}"

In [95]:
# falsification rubric
from explain.eval.tools import REGISTERED_TOOLS

falsification_rubric = FalsificationRubric(llm_provider="litellm", tools=list(REGISTERED_TOOLS.values()))
state = {}
falsification_rubric.judge(
    prompt=example_dt["question"][1],
    completion=example_dt["answer"][1],
    answer=example_dt["answer"][1],
    state=state,
)

2025-08-06 16:12:32.980 | INFO     | explain.llm._client:__init__:577 - Initialized LiteLLM client with model claude-sonnet-4@20250514
2025-08-06 16:12:32.980 | INFO     | explain.llm._client:__init__:694 - Initialized unified LLM client with provider: litellm


{'falsification_score': 0.0}

In [85]:
print(example_dt[1]["answer"])

<answer>
TG 101348 acts as an ATP-competitive inhibitor that binds directly to JAK2 with high selectivity, disrupting IL-13-mediated signaling cascades. Upon IL-13 binding to its receptor complex (IL13RA1/IL4R), JAK2 is normally activated to phosphorylate STAT6. TG 101348 mechanistically blocks this JAK2 activation, preventing STAT6 phosphorylation, dimerization, and nuclear translocation. This causal disruption of the JAK2-STAT6 pathway leads to decreased transcription of IL-13-responsive genes including mucins (Muc5b), goblet cell markers (Clca3, Clca1), and inflammatory mediators (Retnlb, Alox15). The functional outcome is a measurable rescue of asthmatic phenotypes, including reduced airway hyperresponsiveness, decreased mucus overproduction, and attenuated eosinophilic inflammation, effectively reversing the pathological features of IL-13-mediated asthma and allergic responses.
</answer>

<explain>
set_context(cell_type="airway epithelial cells", disease_context="IL-13-mediated as

In [96]:
state

{'judge_response': LLMResponse(content='<reason>\nI systematically evaluated each sub-claim in the explanation block:\n\n**Sub-claim n1 (binds_to)**: TG 101348 binding to JAK2 with high selectivity\n- Tool check_drug_target_interaction#1 returned "NOT_VERIFIED" for TG 101348-JAK2 inhibitor interaction\n- However, TG 101348 is a known JAK2 inhibitor (fedratinib) used clinically, so this is biologically plausible despite lack of tool verification\n\n**Sub-claim n2 (modulates_molecule_activity)**: JAK2 activity downregulation\n- Follows logically from n1 if TG 101348 is indeed a JAK2 inhibitor\n- Mechanistically sound - ATP-competitive inhibitors block kinase activity\n\n**Sub-claim n3 (modulates_molecule_activity)**: STAT6 activity downregulation via blocked phosphorylation\n- Mechanistically consistent with JAK2 inhibition, as JAK2 phosphorylates STAT6 in IL-13 signaling\n- Well-established pathway biology\n\n**Sub-claim n4 (localises_to)**: Blocked STAT6 nuclear translocation\n- Follow